In [ ]:
from pathlib import Path
import ast
import json
import os
import sys

REPO = Path("/workspace/ProtoSAM-repro")
os.chdir(REPO)

assert sys.executable == "/workspace/miniforge3/envs/protosam/bin/python", (
    f"Kernel incorreto: {sys.executable}"
)

BASE_VMAMBA = (
    REPO / "reproducao/multissuporte_tiny_eft2000_kvasir_v1"
)
plano = json.loads(
    (BASE_VMAMBA / "plano.json").read_text(encoding="utf-8")
)
config = json.loads(
    (BASE_VMAMBA / "S00/config.json").read_text(encoding="utf-8")
)

print("=== EXPERIMENTO QUE SERÁ REUTILIZADO ===")
print("Suportes:", len(plano["suportes"]))
print("Queries:", len(config["episodios"]))
print("Encoder:", config["resolucao_encoder"])
print("Canvas SAM:", config["canvas"])
print("SAM:", json.dumps(config["sam"], ensure_ascii=False, indent=2))

assert len(plano["suportes"]) == 20
assert len(config["episodios"]) == 145

print("\n=== CONSTRUTOR E FEATURES DO ALP ===")
arquivo = REPO / "models/grid_proto_fewshot.py"
fonte = arquivo.read_text(encoding="utf-8")
arvore = ast.parse(fonte)
linhas = fonte.splitlines()

for classe in arvore.body:
    if isinstance(classe, ast.ClassDef) and classe.name == "FewShotSeg":
        for metodo in classe.body:
            if isinstance(metodo, (ast.FunctionDef, ast.AsyncFunctionDef)):
                if metodo.name in {
                    "__init__", "get_encoder", "get_cls", "get_features"
                }:
                    print(f"\n--- FewShotSeg.{metodo.name} ---")
                    print("\n".join(
                        linhas[metodo.lineno - 1:metodo.end_lineno]
                    ))

print("\n=== CONSTRUÇÃO DO DINOv2 NO NOTEBOOK ANTERIOR ===")
baseline = REPO / "exp/01_baseline_dinov2_k1.ipynb"

if baseline.is_file():
    nb = json.loads(baseline.read_text(encoding="utf-8"))
    for i, celula in enumerate(nb["cells"]):
        if celula["cell_type"] != "code":
            continue
        texto = "".join(celula["source"])
        if any(termo in texto for termo in (
            "FewShotSeg(",
            "torch.hub.load",
            "load_state_dict",
            "CONFIG_ALP",
            "which_model",
        )):
            print(f"\n--- Célula {i + 1} ---")
            print(texto)
else:
    print("Notebook não encontrado:", baseline)
    print("Notebooks disponíveis relacionados ao baseline:")
    for p in sorted((REPO / "exp").glob("*.ipynb")):
        if any(t in p.name.lower() for t in ("baseline", "dino")):
            print(p)

print("\n=== POSSÍVEIS PESOS LOCAIS DO DINOv2 ===")
pastas = [
    REPO / "pretrained_model",
    Path.home() / ".cache/torch/hub/checkpoints",
]
for pasta in pastas:
    if pasta.exists():
        for p in sorted(pasta.rglob("*")):
            if (
                p.is_file()
                and p.suffix in {".pt", ".pth", ".ckpt"}
                and "dino" in p.name.lower()
            ):
                print(p)

print("\nInspeção concluída. Nenhuma avaliação iniciada.")

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
import random
import gc
import inspect
import subprocess
import importlib.metadata

import numpy as np
import pandas as pd
import torch
from PIL import Image

REPO = Path("/workspace/ProtoSAM-repro")
os.chdir(REPO)

assert sys.executable == "/workspace/miniforge3/envs/protosam/bin/python"
assert torch.cuda.is_available()

for p in (REPO, REPO / "exp/eft_validacao_polipos_v1"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from validar_polipos import medir, conferir_amostra
import models.grid_proto_fewshot as modulo_alp
from models.ProtoSAM import ProtoSAM, ALPNetWrapper, ALPNetInput
from dataloaders.PolypDataset import get_polyp_dataset
from models.segment_anything.utils.transforms import ResizeLongestSide

VMAMBA = REPO / "reproducao/multissuporte_tiny_eft2000_kvasir_v1"
SAIDA = REPO / "reproducao/multissuporte_dinov2_l14_kvasir_v1"

DINO_REPO = REPO / "pretrained_model/hub/facebookresearch_dinov2_main"
DINO_PESOS = REPO / "pretrained_model/hub/checkpoints/dinov2_vitl14_pretrain.pth"

def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def salvar(p, valor):
    p = Path(p)
    tmp = p.with_name(p.name + ".tmp")
    tmp.write_text(
        json.dumps(valor, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, p)

def sha(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def hash_tensor(t):
    return hashlib.sha256(
        t.detach().cpu().contiguous().numpy().tobytes()
    ).hexdigest()

def salvar_csv(df, p):
    p = Path(p)
    tmp = p.with_name(p.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, p)

plano = ler(VMAMBA / "plano.json")
ref = ler(VMAMBA / "S00/config.json")
itens = ref["episodios"]
suportes = plano["suportes"]

assert len(suportes) == 20
assert len(itens) == 145
assert len({s["id"] for s in suportes}) == 20
assert len({r["id"] for r in itens}) == 145
assert (DINO_REPO / "hubconf.py").is_file()
assert DINO_PESOS.is_file()

cfg_sam = dict(ref["sam"])
SAM_PESOS = Path(cfg_sam["sam_pretrained_path"])
assert sha(SAM_PESOS) == ref["checkpoint_sam_sha256"]

gpu_driver = subprocess.check_output(
    [
        "nvidia-smi", "--query-gpu=name,driver_version",
        "--format=csv,noheader",
    ],
    text=True,
).strip()
assert gpu_driver == ref["gpu_driver"], (
    f"GPU/driver diferente do experimento VMamba: {gpu_driver}"
)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
torch.use_deterministic_algorithms(False)

versoes = {
    p: importlib.metadata.version(p)
    for p in ["numpy", "Pillow", "torchvision", "timm", "scipy"]
}
assert str(torch.__version__) == ref["torch"]
assert torch.version.cuda == ref["cuda"]
assert torch.backends.cudnn.version() == ref["cudnn"]
assert versoes == ref["versoes"]

# Confere os arquivos oficiais compartilhados com a avaliação VMamba.
oficiais = [
    "models/grid_proto_fewshot.py",
    "models/alpmodule.py",
    "models/ProtoSAM.py",
    "dataloaders/PolypDataset.py",
    "dataloaders/PolypTransforms.py",
    "models/segment_anything/utils/transforms.py",
]
for relativo in oficiais:
    p = REPO / relativo
    assert sha(p) == ref["fontes_sha256"][str(p)], relativo

cfg_alp = dict(
    which_model="dinov2_l14",
    cls_name="grid_proto",
    proto_grid_size=8,
    lora=0,
    align=False,
    debug=False,
    use_coco_init=False,
    use_pos_enc=False,
    use_slice_adapter=False,
    adapter_layers=3,
    dinov2_loss=False,
)

# Registra também o código local do DINOv2.
fontes_dino = {
    str(p.relative_to(DINO_REPO)): sha(p)
    for p in sorted(DINO_REPO.rglob("*.py"))
}
assert fontes_dino

config = {
    "versao": 1,
    "finalidade": "DINOv2 congelado nos pares da validação multissuporte",
    "modelo": "dinov2_vitl14",
    "sem_treinamento": True,
    "origem_pares": str(VMAMBA),
    "plano_sha256": sha(VMAMBA / "plano.json"),
    "suportes": suportes,
    "episodios": itens,
    "resolucao_encoder": 672,
    "grade_features": [48, 48],
    "canvas": 1024,
    "config_alp": cfg_alp,
    "sam": cfg_sam,
    "alp_inferencia": ref["alp_inferencia"],
    "selecao_sam": ref["selecao_sam"],
    "dino_pesos_sha256": sha(DINO_PESOS),
    "sam_pesos_sha256": sha(SAM_PESOS),
    "fontes_oficiais": {r: sha(REPO / r) for r in oficiais},
    "fontes_dino": fontes_dino,
    "gpu_driver": gpu_driver,
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "versoes": versoes,
    "tf32_matmul": False,
    "tf32_cudnn": True,
}

SAIDA.mkdir(parents=True, exist_ok=True)
if (SAIDA / "config.json").exists():
    assert ler(SAIDA / "config.json") == config, (
        "Configuração mudou. Resultados anteriores preservados."
    )
else:
    salvar(SAIDA / "config.json", config)

print("GPU:", gpu_driver)
print("Pares previstos:", len(suportes) * len(itens))
print("Saída:", SAIDA)
print("Configuração registrada. Avaliação não iniciada.")

In [ ]:
ds_pool, ds_teste_nao_usado = get_polyp_dataset(
    image_size=(1024, 1024),
    sam_trans=ResizeLongestSide(1024),
)
del ds_teste_nao_usado

def caminho(p):
    return str(Path(p).resolve())

mapa = {caminho(p): i for i, p in enumerate(ds_pool.images)}
assert len(mapa) == len(ds_pool.images)

def conferir_arquivo(imagem, mascara, hash_img, hash_mask):
    assert sha(imagem) == hash_img, f"Imagem alterada: {imagem}"
    assert sha(mascara) == hash_mask, f"Máscara alterada: {mascara}"
    indice = mapa[caminho(imagem)]
    assert caminho(ds_pool.gts[indice]) == caminho(mascara)
    return indice

indices_queries = {}
for item in itens:
    indices_queries[item["id"]] = conferir_arquivo(
        item["imagem"], item["mascara"],
        item["imagem_sha256"], item["mascara_sha256"],
    )

indices_suportes = {}
configs_suportes = {}
queries_paths = {caminho(r["imagem"]) for r in itens}

for s in suportes:
    meta = s["meta"]
    assert caminho(meta["imagem"]) not in queries_paths

    indices_suportes[s["id"]] = conferir_arquivo(
        meta["imagem"], meta["mascara"],
        s["imagem_sha256"], s["mascara_sha256"],
    )

    cfg_vm = ler(VMAMBA / s["id"] / "config.json")
    assert cfg_vm["episodios"] == itens
    assert cfg_vm["sam"] == cfg_sam
    assert cfg_vm["gpu_driver"] == gpu_driver
    assert caminho(cfg_vm["suporte"]["imagem"]) == caminho(meta["imagem"])
    assert cfg_vm["suporte"]["imagem_sha256"] == s["imagem_sha256"]
    assert cfg_vm["suporte"]["mascara_sha256"] == s["mascara_sha256"]
    configs_suportes[s["id"]] = cfg_vm

class FewShotSegDinoLocal(modulo_alp.FewShotSeg):
    def get_encoder(self):
        assert self.config["which_model"] == "dinov2_l14"
        assert self.config["lora"] == 0

        self.encoder = torch.hub.load(
            str(DINO_REPO.resolve()),
            "dinov2_vitl14",
            source="local",
            pretrained=False,
        )
        pesos = torch.load(
            DINO_PESOS, map_location="cpu", weights_only=True
        )
        print(self.encoder.load_state_dict(pesos, strict=True))
        del pesos

        lado = max(
            self.image_size // 14, modulo_alp.DEFAULT_FEATURE_SIZE
        )
        self.config["feature_hw"] = [lado, lado]

assert FewShotSegDinoLocal.forward is modulo_alp.FewShotSeg.forward
assert FewShotSegDinoLocal.get_features is modulo_alp.FewShotSeg.get_features
assert FewShotSegDinoLocal.get_cls is modulo_alp.FewShotSeg.get_cls

print("Dados conferidos: 20 suportes e 145 queries.")
print("O conjunto de teste não foi indexado nem avaliado.")
print("Construtor preparado. Modelos ainda não carregados.")

In [ ]:
def executar_dino():
    modelo = wrapper = protosam = None
    tabela_suportes = []

    try:
        modelo = FewShotSegDinoLocal(
            image_size=672,
            pretrained_path=None,
            cfg=dict(cfg_alp),
        ).to("cuda", dtype=torch.float32)
        modelo.eval().requires_grad_(False)

        assert modelo.config["feature_hw"] == [48, 48]
        print("Grade DINOv2:", modelo.config["feature_hw"])
        print("Kernel configurado do ALP:", modelo.cls_unit.kernel_size)

        wrapper = ALPNetWrapper(modelo)
        protosam = ProtoSAM(
            coarse_segmentation_model=wrapper,
            **cfg_sam,
        ).to("cuda", dtype=torch.float32)
        protosam.eval().requires_grad_(False)

        assert not any(p.requires_grad for p in protosam.parameters())
        assert not modelo.training and not protosam.training

        for s in suportes:
            sid = s["id"]
            pasta = SAIDA / sid
            for sub in ("registros", "alp", "sam", "gt"):
                (pasta / sub).mkdir(parents=True, exist_ok=True)

            suporte = ds_pool[indices_suportes[sid]]
            conferir_amostra(suporte)
            assert hash_tensor(suporte["image"]) == (
                configs_suportes[sid]["suporte_tensor_sha256"]
            ), f"Pré-processamento do suporte diferente: {sid}"

            sup_img = suporte["image"].unsqueeze(0).cuda().float()
            sup_mask = suporte["label"].unsqueeze(0).cuda().float()
            registros = []

            print(
                f"\n===== {sid} | {s['meta']['nome']} =====",
                flush=True,
            )

            with torch.no_grad(), torch.autocast("cuda", enabled=False):
                for pos, item in enumerate(itens, start=1):
                    arquivo = pasta / "registros" / f"{item['id']}.json"

                    if arquivo.exists():
                        reg = ler(arquivo)
                        assert reg["item"] == item
                        assert reg["suporte"] == sid
                        for relativo, digest in reg["arquivos_sha256"].items():
                            assert sha(pasta / relativo) == digest
                        registros.append(reg)
                        continue

                    amostra = ds_pool[indices_queries[item["id"]]]
                    conferir_amostra(amostra)
                    query_hash = hash_tensor(amostra["image"])
                    gt = amostra["label"].cpu().numpy() > 0.5

                    # Compara diretamente com o mesmo par avaliado no VMamba.
                    reg_vm = ler(
                        VMAMBA / sid / "antes/registros"
                        / f"{item['id']}.json"
                    )
                    assert reg_vm["item"] == item
                    assert query_hash == reg_vm["query_tensor_sha256"]

                    with Image.open(
                        VMAMBA / sid / "antes/gt" / f"{item['id']}.png"
                    ) as im:
                        assert np.array_equal(gt, np.asarray(im) > 0)

                    qry = amostra["image"].unsqueeze(0).cuda().float()

                    def entrada():
                        return ALPNetInput(
                            support_images=[sup_img],
                            support_labels=[sup_mask],
                            query_images=qry,
                            isval=True,
                            val_wsize=2,
                            show_viz=False,
                            supp_fts=None,
                        )

                    logits = wrapper(entrada())
                    assert tuple(logits.shape) == (1, 2, 1024, 1024)
                    assert torch.isfinite(logits).all().item()
                    alp = (logits.argmax(1)[0] == 1).cpu().numpy()
                    del logits

                    saida, scores = protosam(
                        query_image=qry,
                        coarse_model_input=entrada(),
                        degrees_rotate=0,
                    )
                    sam = saida.detach().cpu().numpy().squeeze()
                    assert sam.shape == (1024, 1024)
                    assert np.isin(sam, [0, 1]).all()
                    sam = sam.astype(bool)

                    hashes = {}
                    for tipo, mascara in (
                        ("alp", alp), ("sam", sam), ("gt", gt)
                    ):
                        p = pasta / tipo / f"{item['id']}.png"
                        Image.fromarray(
                            mascara.astype(np.uint8) * 255
                        ).save(p)
                        hashes[str(p.relative_to(pasta))] = sha(p)

                    reg = {
                        "suporte": sid,
                        "item": item,
                        "query_tensor_sha256": query_hash,
                        "metricas": {
                            "ALP": medir(alp, gt),
                            "SAM": medir(sam, gt),
                        },
                        "arquivos_sha256": hashes,
                    }
                    salvar(arquivo, reg)
                    registros.append(reg)

                    print(
                        f"{sid} {pos}/145 | {Path(item['imagem']).name} | "
                        f"ALP={reg['metricas']['ALP']['Dice']:.4f} | "
                        f"SAM={reg['metricas']['SAM']['Dice']:.4f}",
                        flush=True,
                    )
                    del qry, saida, scores

            assert len(registros) == 145

            por_imagem = []
            for r in registros:
                linha = {
                    "id": r["item"]["id"],
                    "imagem": r["item"]["imagem"],
                }
                for etapa in ("ALP", "SAM"):
                    for metrica in ("Dice", "IoU", "precisao", "recall"):
                        linha[f"{etapa}_{metrica}"] = (
                            r["metricas"][etapa][metrica]
                        )
                por_imagem.append(linha)

            salvar_csv(pd.DataFrame(por_imagem), pasta / "por_imagem.csv")

            resumo = {
                "suporte": sid,
                "n": 145,
                "modelo": "dinov2_vitl14_congelado",
            }
            for etapa in ("ALP", "SAM"):
                resumo[etapa] = {
                    m: float(np.mean([
                        r["metricas"][etapa][m] for r in registros
                    ]))
                    for m in ("Dice", "IoU", "precisao", "recall")
                }
            salvar(pasta / "resumo.json", resumo)

            tabela_suportes.append({
                "suporte": sid,
                "tipo": s["tipo"],
                "imagem": s["meta"]["nome"],
                "ALP_DINO": resumo["ALP"]["Dice"],
                "SAM_DINO": resumo["SAM"]["Dice"],
            })
            salvar_csv(
                pd.DataFrame(tabela_suportes),
                SAIDA / "por_suporte.csv",
            )
            print(
                f"Concluído {sid}: "
                f"ALP={resumo['ALP']['Dice']:.4f} | "
                f"SAM={resumo['SAM']['Dice']:.4f}",
                flush=True,
            )
            del sup_img, sup_mask

        return pd.DataFrame(tabela_suportes)

    finally:
        del protosam, wrapper, modelo
        gc.collect()
        torch.cuda.empty_cache()


tabela_dino = executar_dino()

tabela_vm = pd.read_csv(VMAMBA / "por_suporte.csv")
assert len(tabela_vm) == len(tabela_dino) == 20

comparacao = tabela_vm.merge(
    tabela_dino,
    on=["suporte", "tipo", "imagem"],
    validate="one_to_one",
)
assert len(comparacao) == 20

for etapa in ("ALP", "SAM"):
    comparacao[f"{etapa}_EFT_menos_DINO"] = (
        comparacao[f"{etapa}_EFT"] - comparacao[f"{etapa}_DINO"]
    )

salvar_csv(comparacao, SAIDA / "comparacao_vmamba_dino.csv")
display(comparacao.round(4))

for nome, grupo in (
    ("Todos os 20", comparacao),
    ("Somente os 19 sorteados", comparacao[comparacao["suporte"] != "S00"]),
):
    print(f"\n=== {nome} ===")
    for etapa in ("ALP", "SAM"):
        for modelo_nome, coluna in (
            ("DINOv2", f"{etapa}_DINO"),
            ("VMamba inicial", f"{etapa}_inicial"),
            ("VMamba EFT", f"{etapa}_EFT"),
        ):
            valores = grupo[coluna]
            print(
                f"{etapa} | {modelo_nome}: "
                f"{valores.mean():.4f} ± {valores.std(ddof=1):.4f}"
            )

        delta = grupo[f"{etapa}_EFT_menos_DINO"]
        print(
            f"EFT − DINO: {100 * delta.mean():+.2f} pp; "
            f"EFT maior em {int((delta > 0).sum())}/{len(grupo)} suportes."
        )

print("\nRESULTADOS:", SAIDA)